# 04 — Treinamento e Validação de Modelos

Este notebook treina três modelos de classificação para previsão de churn,
utilizando validação cruzada estratificada (K-Fold).

**Modelos:**
1. Regressão Logística
2. Random Forest
3. LightGBM (ou Gradient Boosting como fallback)

**Pré-requisito:** Execute `03_preprocessamento.ipynb` antes.

---

## 4.1 Importações

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

try:
    from lightgbm import LGBMClassifier
    TEM_LIGHTGBM = True
    print("✓ LightGBM disponível")
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    TEM_LIGHTGBM = False
    print("⚠ LightGBM não disponível — usando GradientBoosting como fallback")

SEED = 42
K_FOLDS = 5
CAMINHO_TREINO = os.path.join("..", "dados", "processado", "dataset_treino.csv")
DIR_MODELOS = os.path.join("..", "modelos")
DIR_METRICAS = os.path.join("..", "resultados", "metricas")
os.makedirs(DIR_MODELOS, exist_ok=True)
os.makedirs(DIR_METRICAS, exist_ok=True)

## 4.2 Carregar Dados de Treino

In [ ]:
df = pd.read_csv(CAMINHO_TREINO)
X = df.drop(columns=["churn"])
y = df["churn"]

print(f"Treino: {X.shape[0]} registros, {X.shape[1]} features")
print(f"Churn: {y.value_counts().to_dict()}")

## 4.3 Definir Modelos

In [ ]:
modelos = {
    "Regressão Logística": LogisticRegression(max_iter=1000, random_state=SEED, C=1.0),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10,
                                             min_samples_split=5, random_state=SEED, n_jobs=-1),
}

if TEM_LIGHTGBM:
    modelos["LightGBM"] = LGBMClassifier(n_estimators=200, max_depth=6,
                                          learning_rate=0.05, random_state=SEED, verbose=-1)
else:
    modelos["Gradient Boosting"] = GradientBoostingClassifier(n_estimators=200, max_depth=6,
                                                               learning_rate=0.05, random_state=SEED)

print(f"Modelos a treinar: {list(modelos.keys())}")

## 4.4 Validação Cruzada Estratificada (K-Fold)

In [ ]:
cv = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

# Usar scorers por string (compatível com todas as versões do scikit-learn)
scoring = {"roc_auc": "roc_auc", "f1": "f1", "accuracy": "accuracy"}

resultados = []

for nome, modelo in modelos.items():
    print(f"\nTreinando: {nome}...")
    cv_results = cross_validate(modelo, X, y, cv=cv, scoring=scoring,
                                 return_train_score=False, n_jobs=-1)
    resultado = {
        "Modelo": nome,
        "AUC-ROC": f"{cv_results['test_roc_auc'].mean():.4f} ± {cv_results['test_roc_auc'].std():.4f}",
        "F1-Score": f"{cv_results['test_f1'].mean():.4f} ± {cv_results['test_f1'].std():.4f}",
        "Acurácia": f"{cv_results['test_accuracy'].mean():.4f} ± {cv_results['test_accuracy'].std():.4f}",
    }
    resultados.append(resultado)
    print(f"  AUC-ROC:  {resultado['AUC-ROC']}")
    print(f"  F1-Score: {resultado['F1-Score']}")
    print(f"  Acurácia: {resultado['Acurácia']}")

df_cv = pd.DataFrame(resultados)
df_cv

## 4.5 Treinar no Dataset Completo e Salvar Modelos

In [ ]:
for nome, modelo in modelos.items():
    modelo.fit(X, y)
    nome_arq = nome.lower().replace(" ", "_") + ".joblib"
    caminho = os.path.join(DIR_MODELOS, nome_arq)
    joblib.dump(modelo, caminho)
    print(f"✓ {nome} salvo em: {caminho}")

# Salvar resultados de CV
df_cv.to_csv(os.path.join(DIR_METRICAS, "validacao_cruzada.csv"), index=False)
print("\n✓ Resultados de CV salvos.")